# Bank Churn (Logistic Regression)

## Introduction

**Dataset**: https://www.kaggle.com/datasets/santoshd3/bank-customers

It is advantageous for banks to know what leads a client towards the decision to leave the company. Churn prevention allows companies to develop loyalty programs and retention campaigns to keep as many customers as possible.

This dataset contains some customers who are withdrawing their account from the bank. With this dataset, we try to analyse and predict which customers have a high chance of leaving the bank.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Import Dataset

In [ ]:
df = pd.read_csv("../data/Churn_Modelling.csv")

df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Exploratory Data Analysis (EDA)

EDA is the first step in a data science or machine learning project. It involves exploring, visualizing, and summarizing your data to understand what is in the dataset, what problems or patterns are there and what steps are needed before modeling. Common Steps are:

- Understand the structure of the dataset
- Check for missing values and duplicate rows
- Visualize distributions
- Check for correlations
- Visualize relationships
- Outlier detection

## Checking for Missing Values and Duplicates

Why do missing values matter?

Most ML algorithms cannot handle undefined values i.e NaN. If there are any missing values in the dataset we typically handle them by:

- **Dropping rows/columns** (if the proportion of missing values is small or irrelevant)

- **Imputing values** using statistical techniques (mean, median, mode, etc.)

In [ ]:
df.isna().any().any()

np.False_

Similiarly, duplicate rows are important. Duplicate rows can skew the learning process. If the same data point appears multiple times, the model may treat it as more important than it really is. We would rather drop these.

In [ ]:
df.duplicated().sum()

np.int64(0)

## Dropping Unnecessary Columns

Some columns are unimportant from the get-go. Some may be are arbitrary identifiers (RowNumber, CustomerID) with no relationship to target. They may also be sensitive personal information (Surname) or, the feature is constant or near constant (for example, if 99% of the participants come from Nigeria then that feature would not be useful for training).

In more practical settings, you may consider whether a feature will be available when making real predictions.

In [ ]:
df.drop(columns=["RowNumber", "CustomerId", "Surname"], inplace=True)

In [ ]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Encoding Categorical Columns

There are two main types of categorical data:
1. **Nominal Categorical Data:** Categories with no inherent order. You cannot rank them meaningfully.

    Examples:
    - Color: Red, Blue, Green
    - Country: Nigeria, Ghana, Kenya
    - Gender: Male, Female


2. **Ordinal Categorical Data:** Categories with a clear, meaningful order.

    Examples:
    - Education level: High school < Bachelor < Master
    - Satisfaction rating: Poor < Average < Good < Excellent
    - Size: Small < Medium < Large

Most ML algorithms only work with numerical inputs. You cannot directly feed text labels like “Red” or “Master’s” into a model. It will not understand them. So, we convert (or *encode*) them into numbers using different strategies depending on the type of category.

The common encoding methods are *label encoding* and *one-hot encoding*.

**Label Encoding**  
    Here, each category is assigned a unique integer that represents the categories (0, 1, 2,...). We use it for *ordinal* data where we want to preserve the order/rank between values. You can also use it if your categorical feature has only two categories (i.e binary) since there is no risk of implying false order (there are literally only two possible values).

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Gender"] = le.fit_transform(df["Gender"])

In [ ]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


**One-Hot Encoding**  
    This creates a new binary (0 or 1) column for each category. For example, if there are 3 categories — France, Germany, Spain — you get 3 new columns. It is used for *nominal* data (no order) where you want to avoid introducing false relationships.

In [ ]:
df = pd.get_dummies(df, columns=["Geography"])

In [ ]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,True,False,False
1,608,0,41,1,83807.86,1,0,1,112542.58,0,False,False,True
2,502,0,42,8,159660.80,3,1,0,113931.57,1,True,False,False
3,699,0,39,1,0.00,2,0,0,93826.63,0,True,False,False
4,850,0,43,2,125510.82,1,1,1,79084.10,0,False,False,True


**Important:** Always apply the same encoding to both training and test sets using encoders fitted only on training data.

## Getting our Features and Targets

Our target (dependent variable) is the "Exited" column. So everything else will be our features (independent variables).

In [ ]:
X = df.drop(columns=["Exited"])
y = df[["Exited"]]

In [ ]:
X

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,True,False,False
1,608,0,41,1,83807.86,1,0,1,112542.58,False,False,True
2,502,0,42,8,159660.80,3,1,0,113931.57,True,False,False
3,699,0,39,1,0.00,2,0,0,93826.63,True,False,False
4,850,0,43,2,125510.82,1,1,1,79084.10,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,True,False,False
9996,516,1,35,10,57369.61,1,1,1,101699.77,True,False,False
9997,709,0,36,7,0.00,1,0,1,42085.58,True,False,False
9998,772,1,42,3,75075.31,2,1,0,92888.52,False,True,False


In [ ]:
y

,Exited
0,1
1,0
2,1
3,0
4,0
...,...
9995,0
9996,0
9997,1
9998,1


## Splitting into train and test sets

When building a machine learning model, we want to know how well it will perform on new, unseen data, not just the data we used to train it. To do that, we split the dataset into two (or more) parts:
1. **Training Set**
    - This is the data the model learns from.

2. **Test Set**
    - This is held back from the model during training.
    - It’s used to evaluate the model’s performance on unseen data.
    - Helps check for *overfitting* — when a model does well on training data but fails to generalize.

Typically, we use a split of **80-20**. 80% of the data for the training set and the remaining 20% for the test set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

In [ ]:
X_train

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
9254,686,1,32,6,0.00,2,1,1,179093.26,True,False,False
1561,632,1,42,4,119624.60,2,1,1,195978.86,False,True,False
1670,559,1,24,3,114739.92,1,1,0,85891.02,False,False,True
6087,561,0,27,9,135637.00,1,1,0,153080.40,True,False,False
6669,517,1,56,9,142147.32,1,0,0,39488.04,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
5734,768,1,54,8,69712.74,1,1,1,69381.05,True,False,False
5191,682,0,58,1,0.00,1,1,1,706.50,True,False,False
5390,735,0,38,1,0.00,3,0,0,92220.12,True,False,False
860,667,1,43,8,190227.46,1,1,0,97508.04,True,False,False


## Scaling Numerical Features

In [ ]:
from sklearn.preprocessing import MinMaxScaler

numerical_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]

scaler = MinMaxScaler()

X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [ ]:
X_train

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
9254,0.672,1,0.189189,0.6,0.000000,0.333333,1,1,0.895550,True,False,False
1561,0.564,1,0.324324,0.4,0.476786,0.333333,1,1,0.980029,False,True,False
1670,0.418,1,0.081081,0.3,0.457317,0.000000,1,0,0.429261,False,False,True
6087,0.422,0,0.121622,0.9,0.540606,0.000000,1,0,0.765408,True,False,False
6669,0.334,1,0.513514,0.9,0.566554,0.000000,0,0,0.197107,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
5734,0.836,1,0.486486,0.8,0.277853,0.000000,1,1,0.346662,True,False,False
5191,0.664,0,0.540541,0.1,0.000000,0.000000,1,1,0.003084,True,False,False
5390,0.770,0,0.270270,0.1,0.000000,0.666667,0,0,0.460925,True,False,False
860,0.634,1,0.337838,0.8,0.758186,0.000000,1,0,0.487381,True,False,False


## Training the Model

Here, we simply initialize our model and feed it our training set.

In [ ]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression()
model.fit(X_train, y_train)

## Evaluating the Model

Once you have trained a classification model, you need to measure how well it performs on unseen data (test set).

You do this using evaluation metrics that tell you how accurate or useful your predictions are. Especially for real-world decisions like spam detection, fraud detection, disease diagnosis, etc.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score


y_pred = model.predict(X_test)

In [ ]:
X_test

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
6252,0.492,1,0.189189,0.3,0.385452,0.333333,0,0,0.208616,False,True,False
4684,0.546,1,0.337838,0.1,0.000000,0.333333,1,1,0.731883,True,False,False
1731,0.502,0,0.351351,0.4,0.000000,0.333333,1,0,0.292531,False,False,True
4742,0.312,1,0.554054,0.8,0.474902,0.333333,1,1,0.853458,False,True,False
4521,0.420,0,0.121622,0.7,0.498194,0.000000,1,1,0.573241,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
6412,0.504,0,0.472973,0.5,0.391668,0.000000,0,1,0.224875,False,True,False
8285,0.518,1,0.094595,1.0,0.000000,0.000000,0,1,0.549353,True,False,False
7853,0.760,0,0.391892,0.7,0.000000,0.000000,1,0,0.166515,True,False,False
1095,0.684,1,0.148649,0.4,0.000000,0.000000,1,0,0.383558,True,False,False


In [ ]:
print(y_pred)

[0 0 0 ... 0 0 0]


In [ ]:
print(y_test)

      Exited
6252       0
4684       0
1731       0
4742       0
4521       0
...      ...
6412       1
8285       0
7853       1
1095       1
6929       1

[2000 rows x 1 columns]


| Cell Type               | Meaning                                  |
| ----------------------- | ---------------------------------------- |
| **TP** (True Positive)  | Model correctly predicted **Positive**   |
| **TN** (True Negative)  | Model correctly predicted **Negative**   |
| **FP** (False Positive) | Model incorrectly predicted **Positive** |
| **FN** (False Negative) | Model incorrectly predicted **Negative** |

Say we test some people for the presence of a disease. Some of these people have the disease, and our test correctly says they are positive. They are called true positives (TP). Some have the disease, but the test incorrectly claims they don't. They are called false negatives (FN). Some don't have the disease, and the test says they don't – true negatives (TN). Finally, there might be healthy people who have a positive test result – false positives (FP). These can be arranged into a 2×2 **confusion matrix**, conventionally with the test result on the vertical axis and the actual condition on the horizontal axis.

|                      | **Predicted Positive** | **Predicted Negative** |
|----------------------|------------------------|------------------------|
| **Actual Positive**  | True Positive (TP)     | False Negative (FN)    |
| **Actual Negative**  | False Positive (FP)    | True Negative (TN)     |

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[1550   57]
 [ 318   75]]


The **accuracy score** is a metric used to evaluate the performance of a model by measuring the proportion of correct predictions made out of the total predictions.

$$
\text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Number of Predictions}}
$$
or

$$
\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}
$$

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

accuracy

0.8125

The **precision score** is a metric used in machine learning to measure the proportion of true positive predictions among all positive predictions made by a model. In other words, how many predicted positives were actually correct.

$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}
$$

It is important when false positives are costly e.g., flagging someone as a fraudster when they are not.

In [ ]:
precision = precision_score(y_test, y_pred)

precision

0.5681818181818182

The **recall score** is a measure of how many actual positives did the model catch.

$$
\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}
$$
It is important when false negatives are costly e.g., missing a cancer diagnosis.

In [ ]:
recall = recall_score(y_test, y_pred)

recall

0.19083969465648856

The **F1 score** is a performance metric used in machine learning that represents the harmonic mean of precision and recall. It is for when you care about both precision and recall.

$$
\text{F1 Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

In [ ]:
f1 = f1_score(y_test, y_pred)

f1

0.2857142857142857

## Extra: Overfitting and Underfitting

When training a machine learning model, our goal is not just to make accurate predictions on the training data — but also to ensure that it performs well on unseen data. This is called **generalization**. Machine Learning models are "good" if:
- It learns patterns effectively from the training data.
- It generalizes well to new, unseen data.

Two common problems that prevent generalization are:
1. Overfitting
2. Underfitting

**Overfitting** happens when a model learns too much from the training data, including details that don’t matter (like noise or outliers). As a result, the model works great on training data but fails when tested on new data.Overfitting models are like students who memorize answers instead of understanding the topic. They do well in practice tests (training) but struggle in real exams (testing). Overfitting can becaused by:
- High variance and low bias.
- The model is too complex.
- The size of the training data.

**Underfitting** is the opposite of overfitting. It happens when a model is too simple to capture what is going on in the data. In this case, the model doesn’t work well on either the training or testing data.
Underfitting models are like students who don’t study enough. They don’t do well in practice tests or real exams. Note: The underfitting model has High bias and low variance. Reasons for Underfitting include:
- The model is too simple, So it may be not capable to represent the complexities in the data.
- The input features which is used to train the model is not the adequate representations of underlying factors influencing the target variable.
- The size of the training dataset used is not enough.
- Excessive regularization are used to prevent the overfitting, which constraint the model to capture the data well.
- Features are not scaled.

In [ ]:
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print(f"Training Accuracy: {train_acc}")
print(f"Test Accuracy: {test_acc}")

Training Accuracy: 0.811875
Test Accuracy: 0.8125


From the accuracy scores of the training and test set we can say that our model is *neither overfitting nor underfitting*.